# Part 1: EDA of Assigned Dataset

**Goal:** Work through an exploratory data analysis of your assigned ICS/SCADA dataset (WaDi or EPIC).

**Deliverables (due next week):**
1. This completed notebook — be ready to walk through it
2. A written dataset description (template at the bottom of this notebook)

**Your dataset lives at:** `/fs/ess/PAS1266/` — find your assigned dataset folder there.

> Before starting, open a terminal and unzip your dataset if needed:
> ```bash
> cd /fs/ess/PAS1266/<your-dataset-folder>
> ls  # see what files are there
> unzip <filename>.zip  # if zipped
> ```

## 1. Research Your Dataset

Before touching the data, go find out what it is. Look for:
- The **original paper** or project page that published the dataset
- A **Kaggle page**, GitHub repo, or other community resource with examples
- Any **data dictionaries** or README files that explain what the columns mean

Write a brief summary below of what you found. Include links.

*TODO: replace with what you found*

**Dataset name:**

**Original source (paper / project page):**

**Community resources (Kaggle, GitHub, blog posts):**

**What the columns represent (from the documentation, not guessing):**

## 2. Using API Documentation

Throughout this notebook you'll see hints like "look up `df.describe()`" or "look up `df.null_count()`". That means go to the library's API reference, find that method, and read what it does, what arguments it takes, and what it returns. This is a core skill — libraries change, new methods get added, and no tutorial covers every function. The API docs are the authoritative source. When you're stuck on how to do something, search the docs before searching Stack Overflow. The docs tell you what the library *actually supports*; Stack Overflow tells you what worked for one person at one point in time.

- **Polars API reference:** https://docs.pola.rs/api/python/stable/reference/index.html
- **Altair API reference:** https://altair-viz.github.io/user_guide/api.html

In [ ]:
import polars as pl
import altair as alt
from pathlib import Path

# Altair limits charts to 5,000 rows by default. For larger datasets,
# either sample your data before plotting (recommended — forces you to
# think about what you're plotting) or disable the limit:
# alt.data_transformers.disable_max_rows()

DATA_DIR = Path("/fs/ess/PAS1266")

## 3. Load the Data

Navigate the shared filesystem to find your dataset file(s). Use `pl.read_csv()` to load it.

**Hints:**
- Run `list(DATA_DIR.iterdir())` in a code cell first to see what's available
- Check delimiter — some datasets use `;` or `\t` instead of `,`
- If the file is large, try loading just the first 1000 rows first with `n_rows=1000` to explore the structure before loading the full dataset

In [ ]:
# TODO: Update the path to point to your specific dataset file
dataset_path = DATA_DIR / "..."  # <-- fill in your path

df = pl.read_csv(dataset_path)  # add separator= if needed

### Sanity check

After loading, run the cell below. You should see a valid shape (rows > 0, columns > 0). If you get an error or 0 rows, double-check your file path and delimiter.

In [ ]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 4. First Look

Answer each question below in its own code cell. These are the basics you should know about any dataset before going deeper.

**Q1:** What are the column names and their data types? (look up `df.schema`)

In [ ]:
# TODO: display the schema (column names + types)

**Q2:** How many numeric columns vs. string/categorical columns are there?

In [ ]:
# TODO: count how many columns are numeric vs. string/categorical
# Hint: df.schema gives you a dict of {column_name: dtype}
# Polars numeric types include pl.Float64, pl.Int64, etc.

**Q3:** What do the first and last 5 rows look like? Do they seem reasonable, or is there header/footer junk?

In [ ]:
# TODO: show the first 5 rows and the last 5 rows

## 5. Summary Statistics

**Q4:** Generate summary statistics for the numeric columns. Look up `df.describe()`.

Look at the output and consider:
- Are there columns where min/max seem unreasonable (sensor readings of -999, etc.)?
- Are there columns with very different scales (one ranges 0–1, another 0–10000)?
- Do any columns have the same value for min and max (constant — useless for analysis)?

In [ ]:
# TODO: generate summary statistics

## 6. Data Quality

**Q5:** How many null (missing) values are in each column? Are there columns that are mostly empty?

Hint: look up `df.null_count()`

In [ ]:
# TODO: check for null values across all columns

**Q6:** Are there any duplicate rows? How many?

Hint: compare `df.shape[0]` to `df.unique().shape[0]`

In [ ]:
# TODO: check for duplicate rows

**Q7:** If your dataset has a label/attack column, what is the class distribution? How imbalanced is it?

Hint: look up `df["column_name"].value_counts()`

In [ ]:
# TODO: find the label/attack column and show its value counts

## 7. Temporal Structure

These are time-series datasets — every row is a reading at a point in time. Understanding the temporal structure is critical because in Part 3 you'll be constructing graphs from **sliding windows** over this timeline. If you don't understand how time works in your data, you can't reason about what a "window" contains.

**Q8:** Find the timestamp column. What format is it in — a datetime, a Unix epoch, a row index, something else? If it's a string, you'll need to parse it. Look up `pl.col("...").str.to_datetime()`.

Hint: check your dataset's documentation from Section 1 — it may tell you the timestamp format.

In [ ]:
# TODO: identify the timestamp column, parse it if needed

**Q9:** What is the time range of the dataset? What is the sampling frequency (time between consecutive rows)? Is it consistent or does it vary?

Hint: `df["timestamp"].diff()` gives the time delta between consecutive rows. Look at its `describe()` to see if the interval is constant.

In [ ]:
# TODO: time range (min, max) and sampling frequency

**Q10:** Are there gaps in the timeline? Gaps mean missing data — a sliding window that spans a gap is mixing two disconnected time periods.

Hint: if sampling frequency is ~1 second, any `diff()` value much larger than that is a gap. Filter for them.

In [ ]:
# TODO: check for gaps in the timeline

**Q11:** When do attacks occur in the timeline? Are they clustered together, spread throughout, or concentrated at the beginning/end? Is there a clear "normal period" vs. "attack period"?

Hint: group by the label column and look at the min/max timestamps per class. Or plot a timeline strip showing label over time.

In [ ]:
# TODO: when do attacks occur? distribution of labels across time

## 8. Visualizations

Create at least 3 charts. Each chart should have a title.

### How Altair works

We're using [Altair](https://altair-viz.github.io/), which takes a **declarative** approach to visualization. That means instead of writing step-by-step drawing instructions ("create a figure, draw a bar here, label this axis"), you describe *what* you want to see and the library figures out *how* to draw it.

Every Altair chart is built from three building blocks:

1. **Data** — your DataFrame. This is the raw material.
2. **Mark** — the visual shape used to represent each row: `mark_bar()`, `mark_point()`, `mark_line()`, `mark_rect()`, etc.
3. **Encoding** — the mapping from columns in your data to visual properties of the mark: which column goes on the x-axis, which on the y-axis, which controls color, size, etc.

You compose these like building blocks:

```python
alt.Chart(df)        # 1. data
   .mark_bar()       # 2. mark — each row becomes a bar
   .encode(          # 3. encoding — map columns to visual properties
       x="column_a",
       y="count()",
       color="column_b"
   )
```

Want a scatter plot instead? Swap `mark_bar()` for `mark_point()`. Want to bin a continuous variable into a histogram? Wrap the encoding: `x=alt.X("column:Q", bin=True)`. The data and encodings stay the same — you're just swapping the visual building block.

The `:Q`, `:N`, `:O`, `:T` suffixes tell Altair the column's type: **Q**uantitative (numbers), **N**ominal (unordered categories), **O**rdinal (ordered categories), **T**emporal (dates/times). Getting this right matters — Altair chooses axis scales and color schemes based on it.

Altair works directly with Polars DataFrames — no conversion needed.

**For more detail:**
- [Altair Basic Statistical Visualization tutorial](https://altair-viz.github.io/getting_started/starting.html) — walks through building charts step by step
- [Altair Example Gallery](https://altair-viz.github.io/gallery/index.html) — find a chart that looks like what you want, read the code
- [Vega-Lite](https://vega.github.io/vega-lite/) — the grammar Altair is built on, if you want to understand the theory

---

**Plot 1:** Distribution of a numeric feature. Pick a sensor column and plot its histogram.

Hint: `mark_bar()` + `alt.X("column:Q", bin=True)` gives you a histogram.

In [ ]:
# TODO: histogram of a numeric feature

**Plot 2:** Class distribution bar chart. Visualize the label/attack column balance.

Hint: `mark_bar()` + `.encode(x="label_column:N", y="count()")` — the `:N` tells Altair the column is nominal (categorical).

In [ ]:
# TODO: bar chart of class/label distribution

**Plot 3:** Correlation heatmap of a subset of numeric features. Pick 8-12 interesting columns — plotting all columns at once will be unreadable.

Hint: Compute the correlation matrix first (select numeric columns → `.to_pandas().corr()`), then reshape it for Altair. Altair needs "long format" for heatmaps — look up `pd.DataFrame.stack()` or `df.melt()`. Then use `mark_rect()` + `.encode(x=, y=, color=)`.

In [ ]:
# TODO: correlation heatmap of a subset of numeric columns

**Plot 4 (open-ended):** Create one more visualization of your choice that reveals something interesting about the data. Some ideas:
- Time series of a sensor reading — `mark_line()` with a timestamp on the x-axis
- Box plots comparing feature distributions across attack types — `mark_boxplot()`
- Scatter plot of two correlated features, colored by label — `mark_point()` + `color="label:N"`

In [ ]:
# TODO: your choice — make it interesting

## 9. Findings

Write 3-5 bullet points summarizing what you learned from this EDA. What stands out? What would you want to investigate further?

*TODO: replace with your findings*

- Finding 1: ...
- Finding 2: ...
- Finding 3: ...
- (optional) Finding 4: ...
- (optional) Finding 5: ...

## 10. Save Cleaned Data

Save a version of your dataset for use in Part 2 (Polars vs Pandas comparison). At minimum, address any data quality issues you found above (nulls, bad rows, etc.). If you identified columns worth dropping, do that here too — but only if you have a reason.

Hint: `df.write_csv()` and `df.write_parquet()` — parquet is faster to reload and smaller on disk.

In [ ]:
# TODO: clean and save the dataset
# 1. Address any data quality issues you found
# 2. Save as parquet — Part 2 will load this file

# df_clean = ...
# df_clean.write_parquet("cleaned_dataset.parquet")

---

## 11. Publish

Your dataset description and EDA walkthrough are a **post on your Quarto site**, not just this notebook. See `quarto-post.md` for the template and steps. The post should include your dataset description and key findings/charts from this notebook.